In [1]:
!mkdir -p week-2
%cd week-2
!pwd

/content/week-2
/content/week-2


In [2]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!ollama --version

In [4]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")

Server says: Ollama is running


In [5]:
!ollama pull llama3.2
!ollama list


NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [6]:
!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'

{"model":"llama3.2","created_at":"2026-11-23T06:30:53.545479522Z","response":"Hello.","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,46864,24748,304,832,2875,11914,13,128009,128006,78191,128007,271,9906,13],"total_duration":13959399397,"load_duration":10174142277,"prompt_eval_count":32,"prompt_eval_cached_count":0,"prompt_eval_duration":3196971000,"eval_count":3,"eval_duration":584853000}

In [2]:
import os, subprocess, time, requests

# 1. Install Ollama if missing (happens after a Colab runtime reset)
if os.system("which ollama > /dev/null 2>&1") != 0:
    print("Installing Ollama...")
    os.system("apt-get -qq install -y zstd && curl -fsSL https://ollama.com/install.sh | sh")

# 2. Start Ollama server in the background
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 3. Wait until the API responds
for _ in range(30):
    try:
        if requests.get("http://localhost:11434").ok:
            print("Ollama server is live.")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server failed to start.")

# 4. Pull llama3.2
!ollama pull llama3.2

Installing Ollama...
Ollama server is live.



In [3]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": "Explain what a REST API is in two sentences.",
        "stream": False
    }
)

print(response.json()["response"])

A REST (Representational State of Resource) API is a type of software architecture that follows a set of guidelines for designing networked applications, emphasizing simplicity, flexibility, and scalability through the use of resource-based interactions. RESTful APIs typically use standard HTTP methods (such as GET, POST, PUT, and DELETE) to manipulate resources, making it easy for developers to build scalable and maintainable applications.


In [4]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # required by the SDK, ignored by Ollama
)

completion = client.chat.completions.create(
    model="llama3.2",
    messages=[
        {"role": "system", "content": "You are a Python coding assistant."},
        {"role": "user", "content": "Write a function to check if a string is a palindrome."}
    ]
)

print(completion.choices[0].message.content)

**Palindrome Checker Function**

The following function checks if a given string is a palindrome, ignoring non-alphanumeric characters and case.

```python
def is_palindrome(s: str) -> bool:
    """
    Checks if a string is a palindrome.

    Args:
    s (str): The input string.

    Returns:
    bool: True if the string is a palindrome, False otherwise.
    """
    s = ''.join(c for c in s if c.isalnum()).lower()  # Remove non-alphanumeric characters and convert to lowercase
    return s == s[::-1]  # Check if the string is the same when reversed
```

**Example Use Cases**
---------------------

```python
print(is_palindrome("A man, a plan, a canal: Panama"))  # True
print(is_palindrome("Not a palindrome"))  # False
print(is_palindrome("Was it a car or a cat I saw"))  # True
```

In this function, we first remove any non-alphanumeric characters from the string and convert it to lowercase. We then check if the resulting string is the same when reversed. If it is, the function returns 

In [5]:
!pip install -q gradio
import gradio as gr
import requests

def chat_with_llama(message, history):
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    response = requests.post(
        "http://localhost:11434/api/chat",
        json={"model": "llama3.2", "messages": messages, "stream": False}
    )
    return response.json()["message"]["content"]

gr.ChatInterface(fn=chat_with_llama, title="Local Llama 3.2 in Colab").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b203fc895c53aa14ff.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
